<a href="https://colab.research.google.com/github/djtheconqueror/home_ai/blob/main/SafeLine_V0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SafeLine V0 — Safety Number Logic Simulator

SafeLine is an early-stage product prototype focused on safer temporary communication.

The goal of this V0 notebook is to simulate a temporary masked-number experience where a user can give out a SafeLine number while out, receive calls/texts normally during Live Mode, and later review, block, mute, or burn contact sessions after leaving the situation.

This version does not use real phone numbers or telecom infrastructure yet. It is a logic simulator for testing the product flow before connecting services like Twilio.

In [1]:
import random
import pandas as pd
from datetime import datetime, timedelta

In [2]:
# SafeLine V0 — basic session state

safeline_state = {
    "safe_number": None,
    "mode": "INACTIVE",  # INACTIVE, LIVE, REVIEW, SHIELD, BURNED
    "created_at": None,
    "expires_at": None,
    "contacts": {}
}

def generate_safe_number():
    area_code = random.choice(["214", "305", "404", "617", "737", "786"])
    prefix = random.randint(200, 999)
    line = random.randint(1000, 9999)
    return f"({area_code}) {prefix}-{line}"

def start_live_session(duration_minutes=120):
    safeline_state["safe_number"] = generate_safe_number()
    safeline_state["mode"] = "LIVE"
    safeline_state["created_at"] = datetime.now()
    safeline_state["expires_at"] = datetime.now() + timedelta(minutes=duration_minutes)
    safeline_state["contacts"] = {}

    print("SafeLine session started.")
    print(f"SafeLine Number: {safeline_state['safe_number']}")
    print(f"Mode: {safeline_state['mode']}")
    print(f"Expires At: {safeline_state['expires_at'].strftime('%Y-%m-%d %H:%M:%S')}")

start_live_session()

SafeLine session started.
SafeLine Number: (617) 963-7118
Mode: LIVE
Expires At: 2026-09-05 21:38:22


In [3]:
# SafeLine V0 — basic session state

safeline_state = {
    "safe_number": None,
    "mode": "INACTIVE",  # INACTIVE, LIVE, REVIEW, SHIELD, BURNED
    "created_at": None,
    "expires_at": None,
    "contacts": {}
}

def generate_safe_number():
    area_code = random.choice(["214", "305", "404", "617", "737", "786"])
    prefix = random.randint(200, 999)
    line = random.randint(1000, 9999)
    return f"({area_code}) {prefix}-{line}"

def start_live_session(duration_minutes=120):
    safeline_state["safe_number"] = generate_safe_number()
    safeline_state["mode"] = "LIVE"
    safeline_state["created_at"] = datetime.now()
    safeline_state["expires_at"] = datetime.now() + timedelta(minutes=duration_minutes)
    safeline_state["contacts"] = {}

    print("SafeLine session started.")
    print(f"SafeLine Number: {safeline_state['safe_number']}")
    print(f"Mode: {safeline_state['mode']}")
    print(f"Expires At: {safeline_state['expires_at'].strftime('%Y-%m-%d %H:%M:%S')}")

start_live_session()

SafeLine session started.
SafeLine Number: (214) 420-5871
Mode: LIVE
Expires At: 2026-09-05 21:38:53


In [5]:
def get_or_create_contact(caller_name, caller_number):
    contact_id = caller_number

    if contact_id not in safeline_state["contacts"]:
        safeline_state["contacts"][contact_id] = {
            "name": caller_name,
            "number": caller_number,
            "texts": 0,
            "calls": 0,
            "status": "ACTIVE",  # ACTIVE, MUTED, BLOCKED, BURNED
            "history": []
        }

    return safeline_state["contacts"][contact_id]


def simulate_contact(caller_name, caller_number, contact_type="text", message=None):
    if safeline_state["mode"] == "INACTIVE":
        print("No active SafeLine session.")
        return

    if safeline_state["mode"] == "BURNED":
        print("SafeLine number has been burned. No contact can come through.")
        return

    contact = get_or_create_contact(caller_name, caller_number)

    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if contact["status"] == "BURNED":
        action = "SESSION DEAD — contact rejected"
    elif contact["status"] == "BLOCKED":
        action = "BLOCKED — contact did not reach user"
    elif contact["status"] == "MUTED":
        action = "MUTED — contact logged silently"
    elif safeline_state["mode"] == "LIVE":
        action = "FORWARDED — user receives normal notification"
    elif safeline_state["mode"] == "SHIELD":
        action = "HELD — contact requires user review"
    elif safeline_state["mode"] == "REVIEW":
        action = "LOGGED — review mode active"
    else:
        action = "LOGGED"

    if contact_type.lower() == "text":
        contact["texts"] += 1
        event = {
            "time": now,
            "type": "TEXT",
            "message": message if message else "",
            "action": action
        }
    elif contact_type.lower() == "call":
        contact["calls"] += 1
        event = {
            "time": now,
            "type": "CALL",
            "message": "",
            "action": action
        }
    else:
        print("contact_type must be 'text' or 'call'.")
        return

    contact["history"].append(event)

    print(f"{event['type']} from {caller_name} ({caller_number})")
    if message:
        print(f"Message: {message}")
    print(f"Action: {action}")

In [6]:
simulate_contact(
    caller_name="Nick",
    caller_number="305-555-4832",
    contact_type="text",
    message="Hey this is Nick from the party"
)

simulate_contact(
    caller_name="Marcus",
    caller_number="214-555-2091",
    contact_type="call"
)

simulate_contact(
    caller_name="Nick",
    caller_number="305-555-4832",
    contact_type="call"
)

TEXT from Nick (305-555-4832)
Message: Hey this is Nick from the party
Action: FORWARDED — user receives normal notification
CALL from Marcus (214-555-2091)
Action: FORWARDED — user receives normal notification
CALL from Nick (305-555-4832)
Action: FORWARDED — user receives normal notification


In [7]:
def end_night():
    if safeline_state["mode"] == "INACTIVE":
        print("No SafeLine session is active.")
        return

    if safeline_state["mode"] == "BURNED":
        print("This SafeLine session has already been burned.")
        return

    safeline_state["mode"] = "REVIEW"

    print("Night ended.")
    print("SafeLine is now in REVIEW MODE.")
    print("New contacts will be logged for review instead of treated like normal Live Mode.")

In [8]:
end_night()

Night ended.
SafeLine is now in REVIEW MODE.
New contacts will be logged for review instead of treated like normal Live Mode.


In [9]:
def contact_summary():
    if not safeline_state["contacts"]:
        print("No contacts logged yet.")
        return pd.DataFrame()

    rows = []

    for number, contact in safeline_state["contacts"].items():
        rows.append({
            "name": contact["name"],
            "number": contact["number"],
            "texts": contact["texts"],
            "calls": contact["calls"],
            "status": contact["status"],
            "total_attempts": contact["texts"] + contact["calls"],
        })

    summary_df = pd.DataFrame(rows).sort_values("total_attempts", ascending=False)
    return summary_df

contact_summary()

,name,number,texts,calls,status,total_attempts
0,Nick,305-555-4832,1,1,ACTIVE,2
1,Marcus,214-555-2091,0,1,ACTIVE,1


In [10]:
simulate_contact(
    caller_name="Nick",
    caller_number="305-555-4832",
    contact_type="text",
    message="You still out?"
)

contact_summary()

TEXT from Nick (305-555-4832)
Message: You still out?
Action: LOGGED — review mode active


,name,number,texts,calls,status,total_attempts
0,Nick,305-555-4832,2,1,ACTIVE,3
1,Marcus,214-555-2091,0,1,ACTIVE,1


In [11]:
def update_contact_status(caller_number, new_status):
    if caller_number not in safeline_state["contacts"]:
        print(f"No contact found for number: {caller_number}")
        return

    allowed_statuses = ["ACTIVE", "MUTED", "BLOCKED", "BURNED"]

    if new_status not in allowed_statuses:
        print(f"Invalid status. Choose one of: {allowed_statuses}")
        return

    safeline_state["contacts"][caller_number]["status"] = new_status

    print(f"Contact {caller_number} is now {new_status}.")


def allow_contact(caller_number):
    update_contact_status(caller_number, "ACTIVE")


def mute_contact(caller_number):
    update_contact_status(caller_number, "MUTED")


def block_contact(caller_number):
    update_contact_status(caller_number, "BLOCKED")


def burn_contact_session(caller_number):
    update_contact_status(caller_number, "BURNED")


def burn_safe_number():
    safeline_state["mode"] = "BURNED"

    print("SafeLine number has been burned.")
    print("No future calls or texts will come through this SafeLine number.")

In [12]:
block_contact("305-555-4832")

simulate_contact(
    caller_name="Nick",
    caller_number="305-555-4832",
    contact_type="text",
    message="Yo why you not responding?"
)

contact_summary()

Contact 305-555-4832 is now BLOCKED.
TEXT from Nick (305-555-4832)
Message: Yo why you not responding?
Action: BLOCKED — contact did not reach user


,name,number,texts,calls,status,total_attempts
0,Nick,305-555-4832,3,1,BLOCKED,4
1,Marcus,214-555-2091,0,1,ACTIVE,1


In [13]:
mute_contact("214-555-2091")

simulate_contact(
    caller_name="Marcus",
    caller_number="214-555-2091",
    contact_type="call"
)

contact_summary()

Contact 214-555-2091 is now MUTED.
CALL from Marcus (214-555-2091)
Action: MUTED — contact logged silently


,name,number,texts,calls,status,total_attempts
0,Nick,305-555-4832,3,1,BLOCKED,4
1,Marcus,214-555-2091,0,2,MUTED,2


In [14]:
burn_contact_session("305-555-4832")

simulate_contact(
    caller_name="Nick",
    caller_number="305-555-4832",
    contact_type="call"
)

contact_summary()

Contact 305-555-4832 is now BURNED.
CALL from Nick (305-555-4832)
Action: SESSION DEAD — contact rejected


,name,number,texts,calls,status,total_attempts
0,Nick,305-555-4832,3,2,BURNED,5
1,Marcus,214-555-2091,0,2,MUTED,2


In [15]:
burn_safe_number()

simulate_contact(
    caller_name="New Person",
    caller_number="404-555-8811",
    contact_type="text",
    message="Hey"
)

SafeLine number has been burned.
No future calls or texts will come through this SafeLine number.
SafeLine number has been burned. No contact can come through.


In [16]:
def safeline_dashboard():
    print("=" * 60)
    print("SAFELINE V0 DASHBOARD")
    print("=" * 60)
    print()
    print(f"SafeLine Number: {safeline_state['safe_number']}")
    print(f"Mode: {safeline_state['mode']}")

    if safeline_state["created_at"]:
        print(f"Created At: {safeline_state['created_at'].strftime('%Y-%m-%d %H:%M:%S')}")
    if safeline_state["expires_at"]:
        print(f"Expires At: {safeline_state['expires_at'].strftime('%Y-%m-%d %H:%M:%S')}")

    print()
    print("Contact Summary:")
    display(contact_summary())

In [17]:
safeline_dashboard()

SAFELINE V0 DASHBOARD

SafeLine Number: (214) 420-5871
Mode: BURNED
Created At: 2026-09-05 19:38:53
Expires At: 2026-09-05 21:38:53

Contact Summary:


,name,number,texts,calls,status,total_attempts
0,Nick,305-555-4832,3,2,BURNED,5
1,Marcus,214-555-2091,0,2,MUTED,2


In [18]:
def contact_history(caller_number):
    if caller_number not in safeline_state["contacts"]:
        print(f"No contact found for number: {caller_number}")
        return pd.DataFrame()

    contact = safeline_state["contacts"][caller_number]

    print(f"Contact History — {contact['name']} ({contact['number']})")
    print(f"Status: {contact['status']}")
    print()

    if not contact["history"]:
        print("No history yet.")
        return pd.DataFrame()

    return pd.DataFrame(contact["history"])

In [19]:
contact_history("305-555-4832")

Contact History — Nick (305-555-4832)
Status: BURNED



,time,type,message,action
0,2026-09-05 19:40:34,TEXT,Hey this is Nick from the party,FORWARDED — user receives normal notification
1,2026-09-05 19:40:34,CALL,,FORWARDED — user receives normal notification
2,2026-09-05 19:41:32,TEXT,You still out?,LOGGED — review mode active
3,2026-09-05 19:42:16,TEXT,Yo why you not responding?,BLOCKED — contact did not reach user
4,2026-09-05 19:42:40,CALL,,SESSION DEAD — contact rejected


In [20]:
print("SAFELINE V0 — PROJECT SUMMARY")
print("=" * 55)
print()
print("Purpose:")
print("SafeLine is a temporary masked-number safety concept designed to help users manage first-contact communication without exposing their real phone number.")
print()
print("Core Idea:")
print("During Live Mode, calls and texts come through normally so the number behaves naturally in social situations.")
print("After the user leaves, they can enter Review Mode and decide who to allow, mute, block, or burn.")
print()
print("Current Capabilities:")
print("- Generate a temporary SafeLine number")
print("- Start a Live Mode session")
print("- Simulate incoming calls and texts")
print("- Track each caller separately")
print("- Display contact summary")
print("- Display contact history")
print("- End the night and enter Review Mode")
print("- Allow, mute, block, or burn individual contact sessions")
print("- Burn the entire SafeLine number")
print()
print("Product Logic:")
print("- Live Mode: contacts are forwarded normally")
print("- Review Mode: contacts are logged for user decision")
print("- Muted: contacts are logged silently")
print("- Blocked: contacts do not reach the user")
print("- Burned Session: one caller loses access")
print("- Burned Number: the full SafeLine alias is disabled")
print()
print("Next Steps:")
print("- Convert simulator into a Streamlit prototype")
print("- Add a simple database for sessions and contacts")
print("- Add user-controlled settings")
print("- Add Twilio integration for real masked-number testing")
print("- Add stronger privacy/security protections")
print()
print("Note:")
print("This V0 notebook does not use real phone numbers. It is a logic simulator for product design and user-flow testing.")

SAFELINE V0 — PROJECT SUMMARY

Purpose:
SafeLine is a temporary masked-number safety concept designed to help users manage first-contact communication without exposing their real phone number.

Core Idea:
During Live Mode, calls and texts come through normally so the number behaves naturally in social situations.
After the user leaves, they can enter Review Mode and decide who to allow, mute, block, or burn.

Current Capabilities:
- Generate a temporary SafeLine number
- Start a Live Mode session
- Simulate incoming calls and texts
- Track each caller separately
- Display contact summary
- Display contact history
- End the night and enter Review Mode
- Allow, mute, block, or burn individual contact sessions
- Burn the entire SafeLine number

Product Logic:
- Live Mode: contacts are forwarded normally
- Review Mode: contacts are logged for user decision
- Muted: contacts are logged silently
- Blocked: contacts do not reach the user
- Burned Session: one caller loses access
- Burned Numb